# Proyecto final

In [1]:
# Librerias estandar
import pandas as pd
import numpy as np
import datasets 
import os
from tqdm.notebook import tqdm
import sklearn 
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore")

# Librerias preprocesamiento
import spacy
from afinn import Afinn
afinn_dict = Afinn(language='en')._dict



# Extra
import utils
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore")

## Datos 

In [2]:
dataset = datasets.load_dataset("sh0416/ag_news")
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 7600
    })
})

In [3]:
train = pd.DataFrame(dataset["train"])
test = pd.DataFrame(dataset["test"])

print(f"Dimensiones train: {train.shape}")
print(f"Dimensiones test: {test.shape}")

print(f"\nTrain:")
train.sample(5)

Dimensiones train: (120000, 3)
Dimensiones test: (7600, 3)

Train:


,label,title,description
38770,2,Mayfield wins pole in Dover,Jeremy Mayfield won the pole in qualifying Fri...
5085,1,Confusion Surrounds Seizure of Iraqi Mosque,"NAJAF, Iraq (Reuters) - The fate of a radical..."
82404,4,Intel readies #39;East Fork #39; digital home...,Intel is gearing up to create a Centrino-style...
23617,4,Cell phone vendors to work on mobile TV,A group of leading mobile phone vendors said F...
3202,2,Cooke Vows to Learn Athens Lesson,Nicole Cooke could ride in four or more Olympi...


## Ejercicio 1

Utilizando AG News, se deberán implementar y comparar varios sistemas y enfoques vistos a lo largo de la 
asignatura. En este ejercicio, la entrada del sistema deberá ser únicamente la columna **description**. La 
columna **title** **no** deberá utilizarse como parte del texto de entrada para clasificar.

In [4]:
# Seleccionamos lo necesario para este ejercicio
X_train = pd.DataFrame(train["description"])
y_train = pd.DataFrame(train["label"])

X_test = pd.DataFrame(test["description"])
y_test = pd.DataFrame(test["label"])

results_dict = {}

### Clasificación mediante lexicón

En primer lugar se va a utilizar un lexicón para hacer un análisis simple de sentimientos, calculando la puntuación de cada texto y clasificando utilizando el cero de umbral. Es el método más sencillo para clasificar y, por tanto, el que peores resultados debería tener. En este caso no se utilizará el conjunto de train, puesto que el modelo no requiere entrenamiento alguno.

In [5]:
# Inicializar spaCy 
try:
    # Intentar cargar el modelo
    print(f"Modelo ya desargado")
    nlp = spacy.load("en_core_web_sm")
    print(f"Modelo cargado")
except OSError:
    # Descargar si es necesario
    print(f"Descargando modelo...")
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")
    print(f"Modelo cargado")
    
nlp = spacy.load("en_core_web_sm")

true_positives = 0

# Iteramos sobre cada texto 
for text in tqdm(X_test, desc="Procesando textos"):
    # Tokenizar con spaCy
    doc = nlp(text)
    
    # Filtrar stopwords y puntuaciones, y obtener los strings limpios
    tokens_limpios = [token.text.lower() for token in doc if not token.is_stop and not token.is_punct]
    
    # Sustituir cada string por su valor en el diccionario de AFINN (0 si no existe)
    valores_afinn = [afinn_dict.get(word, 0) for word in tokens_limpios]
    
    # Pasar a np.array y realizar la suma
    suma = np.array(valores_afinn).sum()
    
    # Condición solicitada
    if suma >= 0:
        true_positives += 1

print(f"Accuracy: {true_positives/len(X_test)}")

results_dict["lexicon"] = true_positives/len(X_test)

Modelo ya desargado
Modelo cargado


Procesando textos: 100%|██████████| 7600/7600 [01:02<00:00, 122.45it/s]

Accuracy: 0.6227631578947368


### Preprocesados textuales

Para los próximos modelos es esencial el pre-procesamiento de los textos. Para ello, se tokenizaran de cuatro formas distintas:

1. Tokenizacion 

2. Tokenización sin puntuaciones.

3. Tokenización sin stopwords.

4. Tokenizacion sin puntuaciones y sin stopwords.

In [5]:
# Ruta de datos 
data_dir = ".data"
os.makedirs(data_dir, exist_ok=True)

# Rutas a los diferents parquet
ficheros_train = [os.path.join(data_dir, f"X_train_{i}.parquet") for i in range(1, 5)]
ficheros_test = [os.path.join(data_dir, f"X_test_{i}.parquet") for i in range(1, 5)]

# Intentamos leer los datos si ya los tenemos guardados 
try:
    print("Intentando cargar las matrices optimizadas desde disco...")
    
    # Leemos y separamos en tokens
    X_train_1 = pd.read_parquet(ficheros_train[0]).iloc[:, 0].str.split()
    X_train_2 = pd.read_parquet(ficheros_train[1]).iloc[:, 0].str.split()
    X_train_3 = pd.read_parquet(ficheros_train[2]).iloc[:, 0].str.split()
    X_train_4 = pd.read_parquet(ficheros_train[3]).iloc[:, 0].str.split()

    X_test_1 = pd.read_parquet(ficheros_test[0]).iloc[:, 0].str.split()
    X_test_2 = pd.read_parquet(ficheros_test[1]).iloc[:, 0].str.split()
    X_test_3 = pd.read_parquet(ficheros_test[2]).iloc[:, 0].str.split()
    X_test_4 = pd.read_parquet(ficheros_test[3]).iloc[:, 0].str.split()
    print("Matrices cargadas y convertidas a listas de tokens correctamente.")

except (FileNotFoundError, Exception):
    print("Ficheros no encontrados o corruptos. Iniciando procesamiento con spaCy...")
    
    nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

    # Generar las matrices (devuelve Series de strings separados por espacios)
    X_train_1, X_train_2, X_train_3, X_train_4 = utils.procesar_matrices(X_train["description"], nlp)
    X_test_1, X_test_2, X_test_3, X_test_4 = utils.procesar_matrices(X_test["description"], nlp)

    print("Guardando matrices procesadas en formato Parquet...")
    # Guardamos el texto plano en el Parquet, es mas eficiente
    pd.DataFrame(X_train_1).to_parquet(ficheros_train[0])
    pd.DataFrame(X_train_2).to_parquet(ficheros_train[1])
    pd.DataFrame(X_train_3).to_parquet(ficheros_train[2])
    pd.DataFrame(X_train_4).to_parquet(ficheros_train[3])

    pd.DataFrame(X_test_1).to_parquet(ficheros_test[0])
    pd.DataFrame(X_test_2).to_parquet(ficheros_test[1])
    pd.DataFrame(X_test_3).to_parquet(ficheros_test[2])
    pd.DataFrame(X_test_4).to_parquet(ficheros_test[3])

    # Convertir las Series de strings a listas de tokens
    X_train_1, X_train_2, X_train_3, X_train_4 = X_train_1.str.split(), X_train_2.str.split(), X_train_3.str.split(), X_train_4.str.split()
    X_test_1, X_test_2, X_test_3, X_test_4 = X_test_1.str.split(), X_test_2.str.split(), X_test_3.str.split(), X_test_4.str.split()
    print("Proceso finalizado con éxito.")

Intentando cargar las matrices optimizadas desde disco...
Matrices cargadas y convertidas a listas de tokens correctamente.


In [6]:
X_train_dict = {
    "1": X_train_1,
    "2": X_train_2,
    "3": X_train_3,
    "4": X_train_4
}

X_test_dict = {
    "1": X_test_1,
    "2": X_test_2,
    "3": X_test_3,
    "4": X_test_4
}

In [7]:
print(f"Vamos a ver, para el primer documento de train, la diferencia de numero de tokens:\n"
      f" - Tokenizacion simple: {len(X_train_1[0])}\n"
      f" - Tokenizacion sin puntuaciones: {len(X_train_2[0])}\n"
      f" - Tokenizacion sin stopwords: {len(X_train_3[0])}\n"
      f" - Tokenizacion sin ambas: {len(X_train_4[0])}\n")

Vamos a ver, para el primer documento de train, la diferencia de numero de tokens:
 - Tokenizacion simple: 20
 - Tokenizacion sin puntuaciones: 14
 - Tokenizacion sin stopwords: 16
 - Tokenizacion sin ambas: 10



Hecho esto, ahora se tienen 4 dataframes por cada tipo de tokenización. La idea es comparar el desempeño de estos cuatro pre-procesamientos a la hora de clasificar. Se utilizarán tanto para la parte de representaciones dispersas y densas (machine learning tradicional) como para fine-tuninig (transformers).

### Representaciones dispersas y densas

Una vez se tienen los tokens hechos toca, mediante diferentes métodos de representacion, clasificar los textos mediante Naives Bayes, Regresión Logística y SVM. Adicionalmente, solo para representaciones densas, se va a probar la LSTM.

#### Dispersas

In [8]:
# Inicializar los diccionarios donde se guardarán las matrices BoW
X_train_BoW = {}
X_test_BoW = {}

# Iterar sobre las 4 configuraciones de tus datos
for key in tqdm(X_train_dict.keys(), desc="Vectorizando matrices BoW"):
    # Instanciar el vectorizador configurado para listas de tokens
    vectorizer = sklearn.feature_extraction.text.CountVectorizer(analyzer=lambda x: x, lowercase=False)
    
    # AJUSTAR y transformar con el conjunto de TRAIN
    X_train_BoW[key] = vectorizer.fit_transform(X_train_dict[key])
    
    # TRANSFORMAR el conjunto de TEST usando solo el vocabulario aprendido de TRAIN
    X_test_BoW[key] = vectorizer.transform(X_test_dict[key])

Vectorizando matrices BoW:   0%|          | 0/4 [00:00<?, ?it/s]

In [11]:
# Definimos hiperparametros para los ditintos modelos
models_and_params = {
    "NaiveBayes": {
        "model": MultinomialNB(),
        "params": {"alpha": [0.1, 1.0]}
    },
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=200, solver="saga", tol=0.1), 
        "params": {"C": [0.1, 1.0]}
    },
    "SVM": {
        # dual=False es más rápido cuando n_samples > n_features
        "model": LinearSVC(max_iter=500, dual=False, tol=1e-3), 
        "params": {"C": [0.1, 1.0]}
    }
}

# Diccionario para almacenar los mejores modelos resultantes
mejores_modelos = {}

# Bucle externo: iterar sobre cada matriz de BOW
for matrix_key in tqdm(X_train_BoW.keys(), desc="Evaluando configuraciones de texto"):
    X_train = X_train_BoW[matrix_key]
    
    # El target 'y_train' debe pasarse como array unidimensional
    y_train_flat = y_train.values.ravel()
    
    mejores_modelos[matrix_key] = {}
    
    # Bucle interno: itera sobre los modelos definidos
    for model_name, config in tqdm(models_and_params.items(), desc=f"GridSearch en Matriz {matrix_key}", leave=False):
        
        # GridSearch sencillo con validación cruzada de 3 pliegues (cv=3) para ahorrar tiempo
        grid = GridSearchCV(
            estimator=config["model"],
            param_grid=config["params"],
            cv=3,
            scoring="accuracy",
            n_jobs=-1 # Usa todos los núcleos disponibles
        )
        
        # Entrenar la búsqueda en la matriz actual
        grid.fit(X_train, y_train_flat)
        
        # Guardar el mejor modelo entrenado de esta combinación
        mejores_modelos[matrix_key][model_name] = grid.best_estimator_
        
        tqdm.write(f"-> Matriz {matrix_key} | {model_name} | Mejores Params: {grid.best_params_} | Mejor CV Accuracy: {grid.best_score_:.4f}")
    

Evaluando configuraciones de texto:   0%|          | 0/4 [00:00<?, ?it/s]

GridSearch en Matriz 1:   0%|          | 0/3 [00:00<?, ?it/s]

-> Matriz 1 | NaiveBayes | Mejores Params: {'alpha': 0.1} | Mejor CV Accuracy: 0.8849
-> Matriz 1 | LogisticRegression | Mejores Params: {'C': 1.0} | Mejor CV Accuracy: 0.8703
-> Matriz 1 | SVM | Mejores Params: {'C': 0.1} | Mejor CV Accuracy: 0.8861


GridSearch en Matriz 2:   0%|          | 0/3 [00:00<?, ?it/s]

-> Matriz 2 | NaiveBayes | Mejores Params: {'alpha': 0.1} | Mejor CV Accuracy: 0.8852
-> Matriz 2 | LogisticRegression | Mejores Params: {'C': 1.0} | Mejor CV Accuracy: 0.8796
-> Matriz 2 | SVM | Mejores Params: {'C': 0.1} | Mejor CV Accuracy: 0.8831


GridSearch en Matriz 3:   0%|          | 0/3 [00:00<?, ?it/s]

-> Matriz 3 | NaiveBayes | Mejores Params: {'alpha': 1.0} | Mejor CV Accuracy: 0.8871
-> Matriz 3 | LogisticRegression | Mejores Params: {'C': 1.0} | Mejor CV Accuracy: 0.8728
-> Matriz 3 | SVM | Mejores Params: {'C': 0.1} | Mejor CV Accuracy: 0.8862


GridSearch en Matriz 4:   0%|          | 0/3 [00:00<?, ?it/s]

-> Matriz 4 | NaiveBayes | Mejores Params: {'alpha': 1.0} | Mejor CV Accuracy: 0.8870
-> Matriz 4 | LogisticRegression | Mejores Params: {'C': 1.0} | Mejor CV Accuracy: 0.8825
-> Matriz 4 | SVM | Mejores Params: {'C': 0.1} | Mejor CV Accuracy: 0.8835


#### Densas

### Fine-tuning de modelos basados en transformers

### Clasificación utilizando LLMs y promting

### Resultados

## Ejercicio 2